# C3 — Un sous-espace par chiffre

| Projet | Étape | Durée | Modules réinvestis |
|---|---|---|---|
| C — Reconnaître des chiffres manuscrits | 3 sur 4 | 3 h | M1, M5, M6, M8 |

In [ ]:
# Style graphique Liouaï : couleurs du logo, axes sobres
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

LIOUAI = {"navy": "#243743", "vert": "#28B78D", "vert_texte": "#167A5C", "gris": "#A1A9AE",
          "gris_texte": "#5C6870", "jaune": "#E5C800", "fond": "#F3F5F6"}
plt.rcParams.update({
    "axes.prop_cycle": plt.cycler(color=[LIOUAI["vert"], LIOUAI["navy"], LIOUAI["gris"],
                                         LIOUAI["jaune"], LIOUAI["vert_texte"], LIOUAI["gris_texte"]]),
    "axes.edgecolor": LIOUAI["gris"], "axes.labelcolor": LIOUAI["navy"],
    "axes.titlecolor": LIOUAI["navy"], "axes.titleweight": "bold",
    "axes.spines.top": False, "axes.spines.right": False,
    "xtick.color": LIOUAI["gris_texte"], "ytick.color": LIOUAI["gris_texte"],
    "text.color": LIOUAI["navy"], "grid.color": "#E1E6E9", "legend.frameon": False,
    "font.family": "sans-serif", "font.sans-serif": ["Roboto", "Helvetica", "Arial", "DejaVu Sans"],
    "figure.dpi": 100,
})
# Dégradé de marque : gris clair -> vert -> navy (et sa version inversée)
if "liouai" not in mpl.colormaps:
    _cmap = LinearSegmentedColormap.from_list("liouai", [LIOUAI["fond"], LIOUAI["vert"], LIOUAI["navy"]])
    mpl.colormaps.register(_cmap)
    mpl.colormaps.register(_cmap.reversed())

## ① Définir la question

Les deux méthodes précédentes sont aux deux extrémités : le chiffre moyen résume trop (une seule image par chiffre), les plus proches voisins ne résument rien (il faut garder toutes les images). Entre les deux, on peut chercher pour chaque chiffre **quelques images de base** qui suffisent à reconstituer, par combinaison linéaire, presque toutes les façons de l'écrire.

**La sous-question de cette étape : peut-on décrire chaque chiffre par un petit sous-espace de $\mathbb{R}^{64}$, et reconnaître une image en cherchant le sous-espace dont elle est la plus proche ?**

**Activité** : Rappelez ce que le module M8 dit de la décomposition en valeurs singulières : que fournissent les premiers vecteurs singuliers d'un nuage de points centré ?

Indiquez vos réponses et justifiez-les.

## ② Estimer avant de calculer

In [ ]:
pari_nombre_directions = 5   # combien de directions par chiffre pour bien reconnaître ? (modifiez)
pari_reussite_sous_espaces = 0.95

## ③ Abstraire

Pour le chiffre $c$, on centre ses images d'apprentissage autour de leur moyenne $\mathbf{m}_c$, on calcule la décomposition en valeurs singulières et l'on garde les $k$ premiers vecteurs singuliers à droite, rangés en colonnes dans une matrice $V_c$ de taille $64 \times k$. Ce sont les **chiffres propres** : les principales façons dont l'écriture de $c$ s'écarte de la moyenne.

Le chiffre $c$ est alors modélisé par l'ensemble des images $\mathbf{m}_c + V_c\,\mathbf{a}$, un sous-espace affine de dimension $k$. Comme les colonnes de $V_c$ sont orthonormées (M6), la projection d'une image $\mathbf{x}$ sur ce sous-espace et le **résidu** s'écrivent

$$
\mathbf{p}_c = \mathbf{m}_c + V_c V_c^{\mathsf{T}} (\mathbf{x} - \mathbf{m}_c), \qquad r_c(\mathbf{x}) = \lVert \mathbf{x} - \mathbf{p}_c \rVert .
$$

On répond le chiffre dont le résidu est le plus petit : celui dont le modèle reconstitue le mieux l'image.

![Schéma : deux sous-espaces, l'un pour les 3 et l'autre pour les 8 ; une image est projetée sur chacun et l'on compare les résidus](../../../assets/schemas/schema-sous-espaces.svg)

Avec $k = 0$, on retrouve exactement la méthode du chiffre moyen. Avec $k$ grand, chaque modèle finit par reconstituer n'importe quelle image, et ne discrimine plus rien.

**Activité** : Pourquoi peut-on écrire la projection avec $V_c^{\mathsf{T}}$ sans résoudre de système ? Combien de nombres faut-il conserver pour 10 chiffres avec $k$ directions chacun ? Comparez avec le chiffre moyen et les plus proches voisins.

Indiquez vos réponses et justifiez-les.

## ④ Calculer

### Les chiffres propres

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from chiffres import charger, decouper, afficher

X, y = charger()
X_reste, y_reste, X_test, y_test = decouper(X, y, part_test=0.25, graine=0)       # test toujours sous scellés
X_app, y_app, X_val, y_val = decouper(X_reste, y_reste, part_test=0.25, graine=1)

def ajuster_sous_espaces(X_app, y_app, k_max=30):
    modeles = []
    for c in range(10):
        Xc = X_app[y_app == c]
        m = Xc.mean(axis=0)
        _, s, Vt = np.linalg.svd(Xc - m, full_matrices=False)
        modeles.append((m, Vt[:k_max].T, s))
    return modeles

modeles = ajuster_sous_espaces(X_app, y_app)
m3, V3, s3 = modeles[3]
afficher(m3, titres=["moyenne des 3"], colonnes=1, taille=1.5)
plt.show()
afficher(V3[:, :10].T, titres=[f"direction {j + 1}" for j in range(10)], taille=1.3, cmap="RdBu_r", symetrique=True)
plt.show()

Les directions ne sont pas des images de chiffres : elles ont des pixels positifs (rouges) et négatifs (bleus). Chacune indique une façon de **modifier** le 3 moyen : épaissir le haut, ouvrir la boucle, incliner le trait…

### Combien de directions faut-il ?

La part de la variation des images de $c$ expliquée par les $k$ premières directions vaut $\left(\sigma_1^2 + \dots + \sigma_k^2\right) / \sum_j \sigma_j^2$ (module M8).

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 3.6))
for c, (m, V, s) in enumerate(modeles):
    part = np.cumsum(s ** 2) / np.sum(s ** 2)
    ax.plot(range(1, 31), part[:30], color=plt.colormaps["tab10"](c), label=str(c))
ax.set_xlabel("nombre de directions k"); ax.set_ylabel("part de variation expliquée")
ax.legend(ncol=5, fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

### Reconnaître par le plus petit résidu

In [ ]:
def residus(X_nouveaux, modeles, k):
    R = np.zeros((len(X_nouveaux), 10))
    for c, (m, V, _) in enumerate(modeles):
        Vk = V[:, :k]
        ecart = X_nouveaux - m
        R[:, c] = np.linalg.norm(ecart - ecart @ Vk @ Vk.T, axis=1)
    return R

def predire_sous_espaces(X_nouveaux, modeles, k):
    return np.argmin(residus(X_nouveaux, modeles, k), axis=1)

ks = [0, 1, 2, 3, 5, 8, 10, 15, 20, 30]
reussites = [np.mean(predire_sous_espaces(X_val, modeles, k) == y_val) for k in ks]

fig, ax = plt.subplots(figsize=(7.5, 3.6))
ax.plot(ks, reussites, "o-", color=LIOUAI["navy"])
ax.set_xlabel("directions par chiffre k (k = 0 : chiffre moyen)"); ax.set_ylabel("réussite (validation)")
ax.grid(alpha=0.3); plt.tight_layout(); plt.show()
print(" ".join(f"k={k}: {r:.3f}" for k, r in zip(ks, reussites)))

### Voir une reconstitution

In [ ]:
k = 10
image = X_val[0]
R = residus(image[None, :], modeles, k)[0]
reconstitutions = []
for c, (m, V, _) in enumerate(modeles):
    Vk = V[:, :k]
    reconstitutions.append(m + (image - m) @ Vk @ Vk.T)
afficher(np.vstack([image, reconstitutions]),
         titres=[f"image ({y_val[0]})"] + [f"modèle {c} : {R[c]:.0f}" for c in range(10)], colonnes=11, taille=1.25)
plt.show()

Chaque modèle tente de reconstituer l'image avec ses propres directions. Le bon modèle y parvient ; les autres produisent une image floue qui ressemble à leur propre chiffre.

### Une vue d'ensemble en deux dimensions

Une analyse en composantes principales de toutes les images d'apprentissage (M8) permet de voir le nuage de $\mathbb{R}^{64}$ projeté sur un plan.

In [ ]:
moyenne_globale = X_app.mean(axis=0)
_, _, Vt = np.linalg.svd(X_app - moyenne_globale, full_matrices=False)
P = (X_app - moyenne_globale) @ Vt[:2].T
fig, ax = plt.subplots(figsize=(6.5, 5.5))
for c in range(10):
    couleur = plt.colormaps["tab10"](c)
    ax.scatter(P[y_app == c, 0], P[y_app == c, 1], s=6, color=couleur, alpha=0.5)
    ax.text(*np.median(P[y_app == c], axis=0), str(c), fontsize=15, weight="bold", color=couleur,
            ha="center", va="center", bbox=dict(facecolor="white", alpha=0.7, edgecolor="none", pad=1))
ax.set_xlabel("composante 1"); ax.set_ylabel("composante 2")
plt.tight_layout(); plt.show()

## ⑤ Interpréter

In [ ]:
k_choisi = ks[int(np.argmax(reussites))]
print(f"Vos paris : {pari_nombre_directions} directions, {pari_reussite_sous_espaces:.0%} de réussite")
print(f"Meilleur résultat en validation : k = {k_choisi}, {max(reussites):.1%}")
print(f"Mémoire à conserver avec k = {k_choisi} : {10 * 64 * (k_choisi + 1)} nombres "
      f"(chiffre moyen : 640 ; plus proches voisins : {X_app.size})")

**Activité** :

1. Choisissez une valeur de $k$ et justifiez votre choix : la meilleure réussite en validation est-elle le seul critère ? Les écarts entre $k = 8$, $10$ et $15$ sont-ils significatifs, avec 337 images de validation ?
2. Faites un tableau des trois méthodes vues (chiffre moyen, plus proches voisins, sous-espaces) : idée, réussite, mémoire, temps, facilité à expliquer une décision.
3. Le graphique en deux dimensions montre des chiffres qui se chevauchent. Est-ce une preuve qu'on ne peut pas les distinguer ? Qu'est-ce que la projection sur un plan cache ?

Indiquez vos réponses et justifiez-les.

## ⑥ Critiquer et généraliser

- **Le modèle s'applique-t-il ?** Un sous-espace suppose que les écritures d'un chiffre se combinent **linéairement** : la moitié d'un 7 barré plus la moitié d'un 7 simple donne un 7 à moitié barré. Est-ce raisonnable ? Que faire pour un chiffre qui a deux styles très différents ?
- **Une décision expliquable.** Le résidu indique aussi la **confiance** : si les deux plus petits résidus sont presque égaux, la machine hésite. Un système de tri peut alors renvoyer l'enveloppe à un humain.
- **Généraliser.** Écrivez une fonction `predire_avec_rejet(X_nouveaux, modeles, k, seuil)` qui renvoie le chiffre reconnu, ou $-1$ si le rapport entre le plus petit et le deuxième plus petit résidu dépasse `seuil`. Pour quelques seuils, mesurez la part d'images rejetées et la réussite sur les images acceptées.

In [ ]:
def predire_avec_rejet(X_nouveaux, modeles, k, seuil=0.9):
    # À compléter
    ...

## Corrigé

*À n'ouvrir qu'après avoir cherché.*

In [ ]:
def predire_avec_rejet(X_nouveaux, modeles, k, seuil=0.9):
    R = residus(X_nouveaux, modeles, k)
    tries = np.sort(R, axis=1)
    predits = np.argmin(R, axis=1)
    predits[tries[:, 0] / tries[:, 1] > seuil] = -1
    return predits

for seuil in (1.0, 0.95, 0.9, 0.8, 0.7):
    p = predire_avec_rejet(X_val, modeles, 10, seuil)
    acceptes = p >= 0
    print(f"seuil {seuil:.2f} : {np.mean(~acceptes):5.1%} rejetées, réussite sur les acceptées {np.mean(p[acceptes] == y_val[acceptes]):.1%}")

## Bilan — auto-évaluation

Pour chaque compétence travaillée dans ce notebook, choisissez un niveau : **0** non observé · **1** guidé · **2** autonome · **3** transfert.
La cellule enregistre votre bilan dans `progression.json`, à la racine du site.

In [ ]:
import json, datetime, pathlib

bilan = {
    "Comparer les mérites des concepts disponibles": 1,
    "Transformer des constantes en paramètres": 1,
    "Décider si les résultats suffisent pour avancer ou abandonner": 1,
    "Évaluer la faisabilité d'obtenir une réponse utile": 1,
}   # modifiez les niveaux

ici = pathlib.Path.cwd()
racine = next((p for p in [ici, *ici.parents] if (p / "myst.yml").exists()), ici)
fichier = racine / "progression.json"
historique = json.loads(fichier.read_text(encoding="utf-8")) if fichier.exists() else []
historique.append({"notebook": "C3-sous-espaces", "date": datetime.date.today().isoformat(), "niveaux": bilan})
fichier.write_text(json.dumps(historique, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Bilan enregistré ({len(historique)} entrée(s) dans {fichier.name}).")